In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score


TRAIN_PATH = "training-path"
TEST_PATH  = "testing-path"

MODEL_NAME = "distilbert-base-uncased"
SAVE_DIR = "./exported_model"

print("Loading training data...")
train_df = pd.read_csv(TRAIN_PATH)
train_df = train_df[["comment_text", "toxic"]]
train_df["toxic"] = train_df["toxic"].astype(float)

# TRAIN / VALIDATION SPLIT
dataset = Dataset.from_pandas(train_df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_ds = dataset["train"]
val_ds   = dataset["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

print("Loading test data (no labels)...")
test_df = pd.read_csv(TEST_PATH)
test_df = test_df[["comment_text"]]
test_ds = Dataset.from_pandas(test_df)

# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

print("Tokenizing datasets...")
train_ds = train_ds.map(preprocess, batched=True)
val_ds   = val_ds.map(preprocess, batched=True)
test_ds  = test_ds.map(preprocess, batched=True)

train_ds = train_ds.rename_column("toxic", "labels")
val_ds   = val_ds.rename_column("toxic", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask"])

# MODEL
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

# METRICS (FOR TRAINER LOGGING)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.squeeze(logits)
    labels = labels.astype(int)
    probs = 1 / (1 + np.exp(-logits))

    return {
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
    }

# TRAINING ARGUMENTS
training_args = TrainingArguments(
    output_dir="./toxicity_model",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=3e-5,
    logging_steps=100,
    fp16=True,
    report_to="none"
)

# TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# TRAIN
print(" Starting training...")
trainer.train()
print(" Training completed.")

# VALIDATION METRICS (ROC-AUC / PR-AUC)
print("Validation metrics:")
val_metrics = trainer.evaluate()
print(val_metrics)

#  THRESHOLD TUNING (ADD THIS EXACTLY HERE)
print(" Finding optimal decision threshold on validation set...")

preds = trainer.predict(val_ds)
logits = np.squeeze(preds.predictions)
labels = preds.label_ids.astype(int)

probs = 1 / (1 + np.exp(-logits))

thresholds = np.linspace(0.1, 0.9, 81)
best_t, best_f1 = 0, 0

for t in thresholds:
    f1 = f1_score(labels, (probs >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(" Best threshold:", round(best_t, 3))
print(" Best F1:", round(best_f1, 4))

# SAVE MODEL
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f" Model saved to {SAVE_DIR}")

# INFERENCE ON TEST.CSV USING OPTIMAL THRESHOLD
print("🔮 Running inference on test.csv...")

predictions = trainer.predict(test_ds)
logits = np.squeeze(predictions.predictions)
probs = 1 / (1 + np.exp(-logits))

test_df["toxicity_score"] = probs
test_df["toxic_pred"] = (probs >= best_t).astype(int)

test_df.to_csv("test_predictions.csv", index=False)
print(" Predictions saved to test_predictions.csv")

# SANITY CHECK (GPU SAFE)
device = model.device

samples = [
    "You are a stupid idiot",
    "Thank you so much for your help",
    "Go die, nobody likes you",
    "This is a wonderful community"
]

print("\nSanity check:")
for s in samples:
    inputs = tokenizer(s, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logit = model(**inputs).logits.item()
        prob = 1 / (1 + torch.exp(-torch.tensor(logit))).item()

    print(f"{prob:.3f} → {s}")


In [ ]:
!pip install captum

In [ ]:
import torch
import numpy as np
from captum.attr import IntegratedGradients
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# LOAD MODEL + TOKENIZER
MODEL_CHECKPOINT = "/kaggle/working/exported_model"
BASE_MODEL = "distilbert-base-uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.to(device)
model.eval()

# FORWARD FUNCTION (EMBEDDINGS)
def forward_func(inputs_embeds, attention_mask):
    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )
    return outputs.logits.squeeze(-1)

ig = IntegratedGradients(forward_func)

# EXPLANATION FUNCTION
def explain_text(text, n_steps=50):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Get embeddings
    embeddings = model.distilbert.embeddings(input_ids)

    # Baseline = zero embedding
    baseline = torch.zeros_like(embeddings).to(device)

    # Integrated Gradients
    attributions = ig.attribute(
        inputs=embeddings,
        baselines=baseline,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps
    )

    # Sum over embedding dimensions
    attributions = attributions.sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

    # Normalize for readability
    attributions = attributions / torch.norm(attributions)

    return tokens, attributions.detach().cpu().numpy()

# VISUALIZATION
def visualize_attributions(tokens, attributions):
    print("\nToken-level attributions:\n")
    for tok, score in zip(tokens, attributions):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        print(f"{tok:>12s} : {score:+.4f}")

# TEST
examples = [
    "You are a stupid idiot",
    "Thank you so much for your help",
    "Go die, nobody likes you",
    "This is a wonderful community"
]

for text in examples:
    print("=" * 80)
    print("TEXT:", text)
    tokens, scores = explain_text(text)
    visualize_attributions(tokens, scores)
